# CTNSG: CircuitSynth Dual-GPU Generation Pipeline
This notebook utilizes Kaggle's Dual-T4 GPUs to run two independent instances of `vLLM` and heavily parallelize the distillation of the CTNSG Silver-Standard dataset using Local Constrained Decoding.

In [ ]:
%pip uninstall -y pydantic
%pip install vllm ray lightweight-charts tqdm requests tensorboard

In [ ]:
import random
import json
import concurrent.futures
from tqdm import tqdm
import subprocess
import time
import requests
import torch
import sys

# ---------------------------------------------------------------------------
# 1. Extreme Diversity Configurations
# ---------------------------------------------------------------------------
DOMAINS = [
    # --- Artificial Intelligence & Data Pipelines ---
    "Retrieval-Augmented Generation (RAG) Document Processing",
    "Large Language Model Distributed Training & Quantization",
    "Multi-Agent Reinforcement Learning Coordination",
    "Synthetic Data Distillation and Validation",
    "Deepfake and Manipulated Media Detection",
    "Computer Vision 3D Scene Reconstruction",
    "Federated Learning across Edge Devices",
    
    # --- Bioinformatics & Healthcare ---
    "Automated Drug Discovery and Lead Optimization",
    "CRISPR Gene Editing and Off-Target Analysis",
    "Protein Folding and Ligand Binding Simulation",
    "Multi-Modal MRI and X-Ray Diagnostic Triage",
    "Genomic Sequence Alignment and Assembly",
    "Epidemic Spread Modeling and Containment Strategy",
    "Brain-Computer Interface (BCI) Signal Decoding",

    # --- Physics, Space & Earth Sciences ---
    "Global Ocean and Climate Forecasting",
    "Low-Earth Orbit Satellite Network Routing",
    "Spacecraft Orbital Debris Collision Avoidance",
    "Computational Fluid Dynamics (CFD) Emulation",
    "Nuclear Fusion Plasma Boundary Optimization",
    "Wildfire Smoke Detection and Emergency Response",
    "Seismic Wavefield Modeling and Tomography",

    # --- Hardware & Engineering ---
    "Semiconductor Lithography and Yield Simulation",
    "VLSI Logic Synthesis and Hardware Routing",
    "Quantum State Preparation and Error Correction",
    "Computer-Aided Design (CAD) Parametric Modeling",
    "Neuromorphic Spiking Neural Network Compilation",
    "Robotic Assembly Line Material Handling",
    
    # --- Autonomous Systems & Robotics ---
    "Multi-Robot Warehouse Fulfillment Logistics",
    "Autonomous Vehicle Sensor Fusion & Motion Planning",
    "Drone-Based Search and Rescue Operations",
    "Humanoid Robot Dexterous Manipulation Control",
    "Spatio-Temporal Traffic Signal Orchestration",

    # --- Cybersecurity & IT Infrastructure ---
    "Zero-Day Malware Reverse Engineering",
    "Distributed Denial of Service (DDoS) Mitigation",
    "Enterprise Cloud CI/CD Deployment Pipeline",
    "Smart Contract Vulnerability Auditing",
    "6G Telecommunications Network Slicing",
    "Data Center Liquid Cooling and Power Optimization",

    # --- Finance, Legal & Economics ---
    "Algorithmic High-Frequency Arbitrage Trading",
    "Cross-Border Financial Fraud Investigation",
    "Dynamic Pricing and Yield Management",
    "Automated Legal Contract Review and Compliance",
    "Real-Time Bidding and Ad Attribution",

    # --- Media, Gaming & Graphics ---
    "Real-Time Multiplayer Game State Synchronization",
    "Procedural 3D Asset and Texture Generation",
    "Audio-Visual Speech Recognition and Translation",
    "Video Rendering and Ray-Tracing Pipeline",

    # --- Heavy Industry & Supply Chain ---
    "End-to-End E-Commerce Supply Chain Tracking",
    "Agricultural Crop Yield and Irrigation Management",
    "Predictive Maintenance for Manufacturing Equipment",
    "Water Treatment and Desalination Plant Control"
]

CONSTRAINTS = [
    "The workflow must contain deep causal dependencies where at least one critical path is 4 to 5 steps deep (e.g., A -> B -> C -> D), requiring strict sequential execution before parallel branches can merge.",
    "Ensure the graph has exactly 7 nodes and at least one bottleneck node where 3 separate tasks merge.",
    "Create a highly parallelized graph where 4 distinct processing pipelines occur simultaneously before culminating in a final aggregation step.",
    "Generate a deep, linear sequence with no parallel branches.",
    "Include a diamond-shaped dependency layout where a node splits into two parallel tasks, which then merge back into a single node.",
    "Create a structure with 2 distinct entry point nodes that operate completely independently before finally syncing at the final node."
]

DAG_SCHEMA = {
    "type": "object",
    "properties": {
        "goal": {"type": "string"},
        "nodes": {
            "type": "array",
            "items": {"type": "string"}
        },
        "edges": {
            "type": "array",
            "items": {
                "type": "array",
                "items": {"type": "integer"}
            }
        }
    },
    "required": ["goal", "nodes", "edges"],
    "additionalProperties": False
}

def has_cycle(edges_list, num_nodes):
    from collections import defaultdict
    adj = defaultdict(list)
    for src, dst in edges_list:
        adj[src].append(dst)
        
    visited = [0] * num_nodes
    
    def dfs(node):
        if visited[node] == 1:
            return True
        if visited[node] == 2:
            return False
        
        visited[node] = 1
        for neighbor in adj[node]:
            if dfs(neighbor):
                return True
        visited[node] = 2
        return False

    for i in range(num_nodes):
        if visited[i] == 0:
            if dfs(i):
                return True
    return False

def verify_dag(parsed_json):
    nodes = parsed_json.get('nodes', [])
    edges = parsed_json.get('edges', [])
    
    num_nodes = len(nodes)
    if num_nodes < 5 or num_nodes > 15:
        return False
        
    for edge in edges:
        if len(edge) != 2:
            return False
        src, dst = edge
        if src < 0 or src >= num_nodes or dst < 0 or dst >= num_nodes:
            return False
            
    if has_cycle(edges, num_nodes):
        return False
        
    return True

def generate_dynamic_prompt():
    domain = random.choice(DOMAINS)
    constraint = random.choice(CONSTRAINTS)
    return f"""You are an expert AI system architect. Your task is to generate a complex Directed Acyclic Graph (DAG) representing a logical workflow.\n\nDOMAIN: {domain}\nSTRUCTURAL CONSTRAINT: {constraint}\nNODE LIMIT: You must generate between 5 and 15 nodes.\n\nThe goal should be a realistic, 1-sentence user request for the {domain} industry.\nEvery node should represent an action_in_snake_case.\nEdges must be represented as pairs of indices [source, destination], zero-indexed, pointing away from the root.\nStrictly ensure there are NO cycles.\n"""

def worker_task(task_id):
    port = 8000 if task_id % 2 == 0 else 8001
    url = f"http://localhost:{port}/v1/chat/completions"
    
    prompt = generate_dynamic_prompt()
    payload = {
        "model": "cyankiwi/Qwen3.5-9B-AWQ-4bit",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.8,
        "max_tokens": 1024,
        "guided_json": DAG_SCHEMA
    }
    
    try:
        response = requests.post(url, json=payload, timeout=20)
        if response.status_code != 200:
            return None
            
        raw_content = response.json()['choices'][0]['message']['content']
        parsed = json.loads(raw_content)
        
        if verify_dag(parsed):
            parsed['trajectory_id'] = f"kaggle_synth_{task_id}"
            return parsed
            
        return None
    except Exception:
        return None

print("Launching Server A on GPU 0 (Port 8000)...")
log_a = open('vllm_server_a.log', 'w')
server_a = subprocess.Popen(
    "VLLM_ATTENTION_BACKEND=TORCH_SDPA CUDA_VISIBLE_DEVICES=0 python -m vllm.entrypoints.openai.api_server "
    "--model cyankiwi/Qwen3.5-9B-AWQ-4bit "
    "--port 8000 --max-model-len 2048 --gpu-memory-utilization 0.80 --max-num-seqs 32 --enforce-eager",
    shell=True, stdout=log_a, stderr=log_a
)

print("Launching Server B on GPU 1 (Port 8001)...")
log_b = open('vllm_server_b.log', 'w')
server_b = subprocess.Popen(
    "VLLM_ATTENTION_BACKEND=TORCH_SDPA CUDA_VISIBLE_DEVICES=1 python -m vllm.entrypoints.openai.api_server "
    "--model cyankiwi/Qwen3.5-9B-AWQ-4bit "
    "--port 8001 --max-model-len 2048 --gpu-memory-utilization 0.80 --max-num-seqs 32 --enforce-eager",
    shell=True, stdout=log_b, stderr=log_b
)

urls = ["http://localhost:8000/v1/models", "http://localhost:8001/v1/models"]
ready = [False, False]

print("Waiting for both engines to boot (Eager mode bypasses CUDA graphs, should be ~60 seconds)...")
while not all(ready):
    if server_a.poll() is not None:
        raise RuntimeError("Server A crashed! Check vllm_server_a.log for details.")
    if server_b.poll() is not None:
        raise RuntimeError("Server B crashed! Check vllm_server_b.log for details.")
        
    for i, url in enumerate(urls):
        if not ready[i]:
            try:
                res = requests.get(url, timeout=2)
                if res.status_code == 200:
                    ready[i] = True
                    print(f"Server on Port {8000 + i} is ready!")
            except requests.exceptions.RequestException:
                pass
    time.sleep(5)
print("Both GPUs fully armed and ready. Starting generation...")

SAMPLES = 5000
MAX_WORKERS = 64
dataset = []

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(worker_task, i): i for i in range(SAMPLES * 2)}
    
    with tqdm(total=SAMPLES, desc="Distilling via Dual T4") as pbar:
        for fut in concurrent.futures.as_completed(futures):
            res = fut.result()
            if res:
                dataset.append(res)
                pbar.update(1)
            
            if len(dataset) >= SAMPLES:
                break

torch.save(dataset, "arbor_graphs_kaggle.pt")
print(f"Successfully serialized {len(dataset)} valid, highly diverse samples.")

print("Tearing down vLLM servers...")
server_a.terminate()
server_b.terminate()
print("GPU instances successfully released.")


### Verification & Diversity Statistics
Run this final cell to verify the quality and topological diversity of the generated data.

In [ ]:
import collections

print(f"Loading data from arbor_graphs_kaggle.pt...")
dags = torch.load("arbor_graphs_kaggle.pt")
num_samples = len(dags)

unique_goals = set()
node_counts = []
edge_counts = []
action_frequencies = collections.Counter()
exact_graph_structures = set()

for dag in dags:
    goal = dag.get("goal", "")
    unique_goals.add(goal.lower().strip())
    
    nodes = dag.get("node_names", dag.get("nodes", [])) # Support both key styles
    node_counts.append(len(nodes))
    for node in nodes:
        action_frequencies[node] += 1
        
    edges = dag.get("text_edges", dag.get("edges", []))
    edge_counts.append(len(edges))
    
    structure_hash = (tuple(sorted(nodes)), tuple(sorted([tuple(e) for e in edges])))
    exact_graph_structures.add(structure_hash)

print("-" * 50)
print("Kaggle Dual-GPU CircuitSynth Diversity Metrics")
print("-" * 50)
print(f"Total Samples:          {num_samples}")
print(f"Unique Goals:           {len(unique_goals)} ({(len(unique_goals)/num_samples)*100:.2f}%)")
print(f"Unique Graph Topologies:{len(exact_graph_structures)} ({(len(exact_graph_structures)/num_samples)*100:.2f}%)")

avg_nodes = sum(node_counts) / max(1, num_samples)
avg_edges = sum(edge_counts) / max(1, num_samples)

print(f"\nAverage Nodes per DAG:  {avg_nodes:.2f}")
print(f"Average Edges per DAG:  {avg_edges:.2f}")

print("\nTop 10 Most Frequent Actions:")
for action, count in action_frequencies.most_common(10):
    print(f"  - {action}: {count}")
